# Data Cleaning — Final Dataset (train + test)

Milestone: missing values, duplicates, invalid values, outliers, `Date` format, final cleaned dataset.

Both `preprocessed_train` and `preprocessed_test` are cleaned **together and in the same way**. Outlier bounds are computed on **train only** and applied to both, so the two sets stay consistent.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Colab: files next to the notebook. Repo: ../dataset/preprocessed
if Path('preprocessed_test.csv').exists():
    DATA = OUT = Path('.')
else:
    DATA, OUT = Path('../dataset/preprocessed'), Path('../dataset/final')
    OUT.mkdir(exist_ok=True)

train_path = next(p for p in [DATA/'preprocessed_train.csv', DATA/'preprocessed_train.zip'] if p.exists())
train = pd.read_csv(train_path, parse_dates=['Date'])
test  = pd.read_csv(DATA/'preprocessed_test.csv', parse_dates=['Date'])
print(train.shape, test.shape)

(1017155, 30) (41088, 29)


## 1. Missing values and duplicates

In [2]:
for name, d in [('train', train), ('test', test)]:
    print(f'{name}: missing={d.isna().sum().sum()}, duplicate rows={d.duplicated().sum()}, '
          f'duplicate Store+Date={d.duplicated(["Store", "Date"]).sum()}')
print('test duplicate Id:', test['Id'].duplicated().sum())

train: missing=0, duplicate rows=0, duplicate Store+Date=0
test: missing=0, duplicate rows=0, duplicate Store+Date=0
test duplicate Id: 0


No missing values and no duplicates in either set, so no imputation and no rows removed.

## 2. Date validation

In [3]:
def date_mismatches(d):
    dow = d['Date'].dt.dayofweek + 1
    return pd.Series({
        'Year': (d['Date'].dt.year != d['Year']).sum(),
        'Month': (d['Date'].dt.month != d['Month']).sum(),
        'Day': (d['Date'].dt.day != d['Day']).sum(),
        'WeekOfYear': (d['Date'].dt.isocalendar().week.astype(int) != d['WeekOfYear']).sum(),
        'Quarter': (d['Date'].dt.quarter != d['Quarter']).sum(),
        'DayOfWeek': (dow != d['DayOfWeek']).sum(),
        'IsWeekend': ((dow >= 6).astype(int) != d['IsWeekend']).sum(),
    })

print(pd.DataFrame({'train': date_mismatches(train), 'test': date_mismatches(test)}))
print('train:', train['Date'].min().date(), '->', train['Date'].max().date())
print('test :', test['Date'].min().date(), '->', test['Date'].max().date())
assert train['Date'].max() < test['Date'].min()

            train  test
Year            0     0
Month           0     0
Day             0     0
WeekOfYear      0     0
Quarter         0     0
DayOfWeek       0     0
IsWeekend       0     0
train: 2013-01-01 -> 2015-07-31
test : 2015-08-01 -> 2015-09-17


All dates are valid and every date-derived column agrees with `Date`. Test starts right after train ends.

## 3. Invalid / inconsistent values

In [4]:
binary = [c for c in train.columns if set(train[c].unique()) <= {0, 1}]
bad_bin = {c: int((~d[c].isin([0, 1])).sum()) for d in (train, test) for c in binary}
print('binary columns checked:', len(binary), '| invalid values:', sum(bad_bin.values()))

groups = {'StoreType': [c for c in train if c.startswith('StoreType_')],
          'Assortment': [c for c in train if c.startswith('Assortment_')],
          'StateHoliday': [c for c in train if c.startswith('StateHoliday_')],
          'Season': [c for c in train if c.startswith('Season_')]}
for name, d in [('train', train), ('test', test)]:
    multi = {g: int((d[cols].sum(axis=1) > 1).sum()) for g, cols in groups.items()}
    promo2 = int(((d['Promo2'] == 0) & (d['IsPromo2Active'] == 1)).sum())
    print(name, 'multi-category rows:', multi, '| Promo2 off but active:', promo2)

print('train: Sales < 0:', (train['Sales'] < 0).sum(),
      '| closed with Sales > 0:', ((train['Open'] == 0) & (train['Sales'] > 0)).sum(),
      '| open with Sales == 0:', ((train['Open'] == 1) & (train['Sales'] == 0)).sum(),
      '| SalesLog mismatch:', (~np.isclose(train['SalesLog'], np.log1p(train['Sales']), atol=1e-5)).sum())

binary columns checked: 17 | invalid values: 0


train multi-category rows: {'StoreType': 0, 'Assortment': 0, 'StateHoliday': 0, 'Season': 0} | Promo2 off but active: 0
test multi-category rows: {'StoreType': 0, 'Assortment': 0, 'StateHoliday': 0, 'Season': 0} | Promo2 off but active: 0
train: Sales < 0: 0 | closed with Sales > 0: 0 | open with Sales == 0: 0 | SalesLog mismatch: 0


No invalid values or logical conflicts. All-zero in a one-hot group is valid (omitted baseline category).

## 4. Outliers
Checked with the IQR rule on the three continuous (scaled) features. Bounds come from **train only**.

`Sales` is **not** capped: high sales are real (promotions, large stores) and the model target `SalesLog` is already log-transformed.

In [5]:
cont = ['CompetitionDistanceLog_scaled', 'CompetitionOpenMonths_scaled', 'Promo2OpenMonths_scaled']

bounds = {}
for c in cont:
    q1, q3 = train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    bounds[c] = (np.ceil((q1 - 1.5 * iqr) * 1e6) / 1e6, np.floor((q3 + 1.5 * iqr) * 1e6) / 1e6)  # 6 decimals, inside the IQR limits

report = pd.DataFrame(bounds, index=['lower', 'upper']).T
report['train_outliers'] = [int(((train[c] < lo) | (train[c] > hi)).sum()) for c, (lo, hi) in bounds.items()]
report['test_outliers'] = [int(((test[c] < lo) | (test[c] > hi)).sum()) for c, (lo, hi) in bounds.items()]
report['train_%'] = (report['train_outliers'] / len(train) * 100).round(2)
report['test_%'] = (report['test_outliers'] / len(test) * 100).round(2)
report.round(3)

,lower,upper,train_outliers,test_outliers,train_%,test_%
CompetitionDistanceLog_scaled,-2.880,2.954,942,48,0.09,0.12
CompetitionOpenMonths_scaled,-2.768,2.602,11722,864,1.15,2.10
Promo2OpenMonths_scaled,-2.663,2.597,18349,4704,1.80,11.45


Outliers are capped (winsorized) at the train bounds instead of deleting rows, so no observations are lost.

In [6]:
before = pd.DataFrame({'train_max': train[cont].max(), 'test_max': test[cont].max()})
for c, (lo, hi) in bounds.items():
    train[c] = train[c].clip(lo, hi).round(6)
    test[c] = test[c].clip(lo, hi).round(6)
after = pd.DataFrame({'train_max': train[cont].max(), 'test_max': test[cont].max()})
print('Max before:'); print(before.round(3)); print('Max after:'); print(after.round(3))

Max before:
                               train_max  test_max
CompetitionDistanceLog_scaled      2.308     2.308
CompetitionOpenMonths_scaled      11.108    11.145
Promo2OpenMonths_scaled            3.096     3.201
Max after:
                               train_max  test_max
CompetitionDistanceLog_scaled      2.308     2.308
CompetitionOpenMonths_scaled       2.602     2.602
Promo2OpenMonths_scaled            2.597     2.597


## 5. Final validation

In [7]:
feats = [c for c in train.columns if c not in ('Sales', 'SalesLog')]
assert train.isna().sum().sum() == 0 and test.isna().sum().sum() == 0
assert train.duplicated().sum() == 0 and test.duplicated().sum() == 0
assert np.isfinite(train.select_dtypes('number')).all().all() and np.isfinite(test.select_dtypes('number')).all().all()
assert set(feats) == set(test.columns) - {'Id'}
for c, (lo, hi) in bounds.items():
    assert train[c].between(lo, hi).all() and test[c].between(lo, hi).all()
assert pd.api.types.is_datetime64_any_dtype(train['Date']) and pd.api.types.is_datetime64_any_dtype(test['Date'])
print('train', train.shape, '| test', test.shape, '| all checks passed')

train (1017155, 30) | test (41088, 29) | all checks passed


## 6. Export

In [8]:
train.to_csv(OUT / 'final_cleaned_train.csv', index=False)
test.to_csv(OUT / 'final_cleaned_test.csv', index=False)
print('Saved to', OUT.resolve())

Saved to /home/claude/repo/dataset/final


## Summary
| Check | Result | Action |
|---|---|---|
| Missing values | none (train, test) | none needed |
| Duplicates | none | none removed |
| Date | valid, derived columns consistent | ISO `YYYY-MM-DD` |
| Invalid / inconsistent values | none | none needed |
| Outliers | 3 continuous features | IQR capping, bounds from train, applied to train and test |
| `Sales` | high values are real | kept (use `SalesLog`) |